# INUA Sprint 3: Machine Learning Model Training

**Student:** Michelle  
**Project:** INUA - Intelligent Mentorship Platform  
**Institution:** Strathmore University (Final Year Project - BICS Year 4)  
**Algorithms:** Random Forest, AdaBoost, XGBoost  
**Validation Strategy:** 5-Fold Stratified Cross-Validation + 20% Held-Out Test Partition  

---

## 1. Research Objectives
1. **Leakage-Free Preprocessing**: Exclude arbitrary identifiers (`session_id`) and intermediate outcome variables (`effectiveness_index`).
2. **ColumnTransformer Pipeline**: Fit `StandardScaler` on numerical features and `OneHotEncoder` on categorical features strictly on the training partition.
3. **Cross-Validation**: Perform 5-fold stratified cross-validation on 1,600 training instances to evaluate generalization stability.
4. **Serialization**: Save the trained model binaries (`.pkl`) to `ml/saved_models/` for downstream backend integration.

In [ ]:
import os
import sys
import joblib
import pandas as pd
import numpy as np

# Add scripts directory to path
sys.path.append(os.path.abspath(os.path.join('..', 'scripts')))
from preprocessing import run_preprocessing
from train_models import initialize_models, perform_cross_validation

# Execute Preprocessing Pipeline
X_train, X_test, y_train, y_test, feature_names = run_preprocessing(
    csv_path=os.path.join('..', 'data', 'inua_mentorship_dataset.csv'),
    output_dir=os.path.join('..', 'data'),
    models_dir=os.path.join('..', 'saved_models')
)
print(f"Feature space dimension: {len(feature_names)} features")

## 2. Model Initialization & 5-Fold Cross-Validation
We evaluate 3 tree-based ensemble paradigms:
- **Random Forest**: Bagging ensemble reducing variance across de-correlated decision trees.
- **AdaBoost**: Sequential boosting emphasizing misclassified boundary instances.
- **XGBoost**: Gradient boosting with regularized learning objectives (`L1`/`L2` penalty).

In [ ]:
models = initialize_models(random_state=42)
cv_results = perform_cross_validation(models, X_train, y_train, n_splits=5)

cv_df = pd.DataFrame(cv_results).T
cv_df.columns = ['CV Accuracy Mean', 'CV Accuracy Std', 'CV ROC-AUC Mean', 'CV ROC-AUC Std']
cv_df.round(4)

## 3. Final Model Fitting on Training Partition (N=1600)

In [ ]:
trained_models = {}
for name, model in models.items():
    print(f"Training {name}...")
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    # Save model binary
    save_name = name.lower().replace(' ', '_') + ('.pkl' if 'model' in name.lower() else '_model.pkl' if name == 'XGBoost' else '.pkl')
    if name == 'XGBoost':
        save_name = 'xgboost_model.pkl'
    save_path = os.path.join('..', 'saved_models', save_name)
    joblib.dump(model, save_path)
    print(f"Saved {name} -> {save_path}")